# 15. Exposure, replication, and variance decomposition

![Exposure and replication](../images/15_exposure_and_replication.svg)

**Learning goals:** distinguish unique units from repeated exposure, aggregate nested sampling pools for a stated estimand, distinguish randomization, sampling, and analysis units, separate participant and model replication, estimate variance components, prevent group leakage, audit complete cases, and prospectively lock claims.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 15
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Unique sources, rows, and exposures

A source observation can generate several augmented rows, and each row can be revisited for many epochs. These counts affect computation and fitting, but only independently sampled source units expand the population sample. We retain `source_id` through augmentation so duplicates remain auditable.

In [ ]:
n_sources, augmentations_per_source, epochs = 120, 3, 20
source_id = np.repeat(np.arange(n_sources), augmentations_per_source)
n_rows = len(source_id)
total_exposures = epochs * n_rows
assert np.unique(source_id).size == n_sources
assert total_exposures == 7200
print(f"unique sources={n_sources}, augmented rows={n_rows}, training exposures={total_exposures}")

## 2. Nested pools imply a weighting choice

Participants contribute different numbers of observations below. The global row mean estimates the outcome of a randomly selected row, so long recordings dominate. The equal-participant mean first averages within participant and estimates the outcome of a randomly selected participant. Neither is automatically correct. The target population determines the aggregation.

In [ ]:
participant_sizes = np.array([5, 8, 12, 20, 35, 50])
participant_ids = np.repeat(np.arange(len(participant_sizes)), participant_sizes)
participant_means_true = np.array([0.2, 0.3, 0.5, 0.7, 1.0, 1.3])
outcomes = participant_means_true[participant_ids] + rng.normal(0, .15, len(participant_ids))
global_row_mean = outcomes.mean()
observed_participant_means = np.array([outcomes[participant_ids == i].mean()
                                       for i in range(len(participant_sizes))])
equal_participant_mean = observed_participant_means.mean()
assert global_row_mean > equal_participant_mean
print(f"global row mean={global_row_mean:.3f}")
print(f"equal-participant mean={equal_participant_mean:.3f}")

## 3. Randomization, sampling, and analysis units

A randomization unit is independently assigned to treatment, a sampling unit is independently drawn from a target population, and an analysis unit contributes an independent contrast or random effect to uncertainty estimation. They may coincide, but independent sampling is not the same as experimental assignment. If treatment is randomized by participant, the participant is the randomization unit and frames are repeated observations. For equal cluster size $m$ and intraclass correlation $\rho$, the approximate design effect is $1+(m-1)\rho$. Dividing row count by this value gives intuition for effective sample size, but not a replacement for a mixed model or cluster bootstrap.

In [ ]:
m, rho, n_clustered_rows = 20, 0.50, 200
design_effect = 1 + (m - 1) * rho
effective_n = n_clustered_rows / design_effect
assert np.isclose(design_effect, 10.5)
print(f"design effect={design_effect:.1f}; approximate effective n={effective_n:.1f}")

## 4. Participant and model-seed variance are different

We simulate a fully crossed table $Y_{im}=\mu+u_i+w_m+e_{im}$ with participants as rows and independently trained model seeds as columns. Balanced two-way ANOVA mean squares estimate participant, model, and residual components. For a grand mean, the variance contribution is $\sigma_u^2/P+\sigma_w^2/M+\sigma_e^2/(PM)$. This shows which replication axis limits precision.

In [ ]:
P, M = 30, 6
true_sd_participant, true_sd_model, true_sd_residual = 1.0, 0.45, 0.60
u = rng.normal(0, true_sd_participant, (P, 1))
w = rng.normal(0, true_sd_model, (1, M))
e = rng.normal(0, true_sd_residual, (P, M))
Y = 2.0 + u + w + e
grand = Y.mean()
row_mean = Y.mean(axis=1, keepdims=True)
col_mean = Y.mean(axis=0, keepdims=True)
residual = Y - row_mean - col_mean + grand
ms_participant = M * np.sum((row_mean - grand) ** 2) / (P - 1)
ms_model = P * np.sum((col_mean - grand) ** 2) / (M - 1)
ms_residual = np.sum(residual ** 2) / ((P - 1) * (M - 1))
var_participant = max(0.0, (ms_participant - ms_residual) / M)
var_model = max(0.0, (ms_model - ms_residual) / P)
var_residual = ms_residual
grand_mean_variance = var_participant / P + var_model / M + var_residual / (P * M)
assert all(v >= 0 for v in (var_participant, var_model, var_residual))
print("estimated variance components:",
      {"participant": round(var_participant, 3), "model": round(var_model, 3),
       "residual": round(var_residual, 3)})
print(f"estimated grand-mean SE={np.sqrt(grand_mean_variance):.3f}")

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(["participant", "model seed", "residual"],
       [var_participant, var_model, var_residual], color=["#3b82b8", "#8057b5", "#d89b21"])
ax.set(ylabel="estimated variance", title="Different sources require different replication")
plt.tight_layout()
plt.show()

## 5. Leakage, complete cases, and prospective locking

A group split assigns entire participants to train or test before any data-dependent preprocessing. Complete-case analysis retains participants with both conditions, but targets the complete subpopulation if missingness is informative. A prospective analysis plan records all unit definitions, an explicit split map, exact exclusions, bootstrap replicate count and seed, multiplicity family, stopping rule, and an executable sensitivity specification before outcomes are inspected.

In [ ]:
all_participants = np.arange(40)
split_rng = np.random.default_rng(1501)
test_participants = set(split_rng.choice(all_participants, size=10, replace=False).tolist())
train_participants = set(all_participants.tolist()) - test_participants
assert train_participants.isdisjoint(test_participants)

difficulty = rng.normal(size=40)
full_differences = 0.20 - 0.12 * difficulty + rng.normal(0, .05, 40)
observed_intervention = difficulty < 0.8  # difficult units fail more often
observed_differences = np.where(observed_intervention, full_differences, np.nan)
complete_mask = observed_intervention & np.isfinite(observed_differences)
complete_case_mean = observed_differences[complete_mask].mean()
excluded_participants = all_participants[~complete_mask].tolist()
print(f"complete cases={complete_mask.sum()}/40")
print(f"full mean={full_differences.mean():.3f}; complete-case mean={complete_case_mean:.3f}")

assignment_map = {str(pid): ("test" if pid in test_participants else "train")
                  for pid in all_participants}
sensitivity_spec = {
    "procedure": "single_value_imputation_for_missing_participant_contrasts",
    "missing_fill_grid": [-0.10, 0.00, 0.10],
    "summary": "mean_over_all_40_participants",
}

def run_locked_sensitivity(observed, observed_mask, spec):
    results = {}
    for fill in spec["missing_fill_grid"]:
        completed = np.where(observed_mask, observed, fill)
        results[f"fill={fill:+.2f}"] = float(completed.mean())
    return results

sensitivity_results = run_locked_sensitivity(observed_differences, complete_mask, sensitivity_spec)
assert len(sensitivity_results) == 3 and all(np.isfinite(list(sensitivity_results.values())))

locked_plan = {
    "primary_endpoint": "participant_mean_intervention_minus_baseline",
    "units": {"randomization": None, "sampling": "participant",
              "analysis": "participant paired contrast"},
    "group_assignments": assignment_map,
    "split_generation_seed": 1501,
    "exclusion_rules": ["missing either paired condition",
                        "nonfinite participant contrast"],
    "realized_excluded_participants": excluded_participants,
    "interval": {"method": "participant percentile bootstrap",
                 "confidence": 0.95, "replicates": 5000, "seed": 1502},
    "equivalence_margin": 0.10,
    "multiplicity": {"family": ["primary paired contrast"],
                     "method": "none for one locked hypothesis"},
    "stopping_rule": "analyze once after 40 enrolled participants",
    "sensitivity_analysis": sensitivity_spec,
}
assert set(assignment_map.values()) == {"train", "test"}
assert len(assignment_map) == len(all_participants)
print("locked sensitivity means:", {k: round(v, 3) for k, v in sensitivity_results.items()})

## Exercises, generalization limits, and takeaways

1. Increase epochs from 20 to 100. Which counts change, and which do not?
2. Increase model seeds while holding participants fixed. Which grand-mean variance term remains a floor?
3. Split frames instead of participants. Describe the leakage path.
4. Make missingness more likely for small effects and inspect complete-case bias.

**Brief answers:** exposures increase, but unique sources and rows do not. The participant term $\sigma_u^2/P$ remains. Frame splitting lets participant and sequence information cross folds. Informative missingness changes the retained population and can shift the estimated effect.

**Generalization limits:** new frames, sequences, participants, model seeds, devices, and sites are different extrapolations. A design supports only the dimensions it independently samples or deliberately holds out.

**Takeaway:** row count is not evidence count. Track source units, dependence groups, exposure, missingness, and every intended random dimension separately.

## Continue learning

[Previous notebook: 14](14_paired_inference.ipynb) | [Lecture](../lectures/15_exposure_and_replication.md) | [Curriculum](../README.md)